# Candlestick pattern detector — YOLO11n fine-tuning (Colab, T4)

This notebook is the **only** step of the project that needs a GPU. Everything before it
(fetching bars, rendering charts, generating rule-based labels) and everything after it
(detection metrics, the downstream signal study, the demo) runs on CPU from the repo.

## Before you run

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Open the **🔑 Secrets** panel in the left sidebar, add a secret named `HF_TOKEN`
   (same value as your local `.env`), and toggle *Notebook access* on.
3. **Runtime → Run all.**

## What it does

Downloads the rendered dataset from the Hugging Face dataset repo, fine-tunes `yolo11n`
from its pretrained COCO checkpoint, evaluates per class on the held-out **test** split,
and pushes weights plus metrics to the Hugging Face model repo.

## Free-tier resilience

Colab can disconnect mid-run. Training therefore checkpoints to the **model repo on the
Hub**, not to Colab's ephemeral disk, after every epoch. If the session dies, just run the
notebook again: it finds the checkpoint, resumes from that epoch, and keeps its progress.

## What the labels are

Boxes come from TA-Lib's `CDLxxx` rules, **not** from human annotation. The detector is
being trained to imitate a published heuristic. That framing matters for how the final
results should be read, and it is stated in the repo README as well.

## 1. Install dependencies

In [ ]:
%pip install -q ultralytics huggingface_hub

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('\n*** NO GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again. ***')

## 2. Authenticate with Hugging Face

Reads `HF_TOKEN` from the Colab Secrets panel. Nothing is printed.

In [ ]:
import os
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    # Running outside Colab: fall back to an already-set env var.
    assert os.environ.get('HF_TOKEN'), 'set HF_TOKEN in the environment'

login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
api = HfApi()
print('authenticated as', api.whoami()['name'])

## 3. Download the rendered dataset

The dataset is stored as one gzipped tarball per split. Each contains the chart PNGs and
the matching YOLO label files. Splits are **chronological** with an embargo gap, so no
validation or test chart shares a candle with a training chart.

In [ ]:
import tarfile, pathlib
from huggingface_hub import hf_hub_download

DATASET_REPO = 'rohanjain2312/candlestick-pattern-recognition-system-data'
ROOT = pathlib.Path('/content/dataset')
ROOT.mkdir(parents=True, exist_ok=True)

for split in ('train', 'val', 'test'):
    path = hf_hub_download(DATASET_REPO, f'{split}.tar.gz', repo_type='dataset')
    with tarfile.open(path) as tf:
        tf.extractall(ROOT)
    n = len(list((ROOT / 'images' / split).glob('*.png')))
    print(f'{split:>5}: {n} images')

## 4. Write `data.yaml`

Class ids must match the order in `src/config.py`. They are written out literally here so
the notebook is self-contained and cannot silently disagree with the label files.

In [ ]:
import yaml

CLASSES = ['Hammer', 'ShootingStar', 'BullishEngulfing', 'BearishEngulfing', 'MorningStar', 'EveningStar', 'Doji', 'Harami']
data_yaml = {
    'path': str(ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(CLASSES),
    'names': {i: c for i, c in enumerate(CLASSES)},
}
yaml_path = ROOT / 'data.yaml'
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(yaml_path.read_text())

## 5. Resume from the Hub, if a previous run got cut off

Looks for `last.pt` in the model repo. If it is there, training continues from that epoch;
otherwise we start from the pretrained `yolo11n` COCO checkpoint.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = 'rohanjain2312/candlestick-pattern-recognition-system-yolo'
RUN_DIR = pathlib.Path('/content/runs/detect/train')
RUN_DIR.mkdir(parents=True, exist_ok=True)

resume_from = None
try:
    ckpt = hf_hub_download(MODEL_REPO, 'last.pt')
    target = RUN_DIR / 'weights' / 'last.pt'
    target.parent.mkdir(parents=True, exist_ok=True)
    import shutil; shutil.copy2(ckpt, target)
    resume_from = str(target)
    print('found an interrupted run; will resume from', resume_from)
except Exception as exc:
    # No checkpoint yet is the normal first-run case, not an error.
    print('no checkpoint on the Hub, starting fresh  (%s)' % type(exc).__name__)

## 6. Fine-tune

**Augmentation choices matter here.** Horizontal flip and rotation are *disabled*: flipping
a candlestick chart reverses the direction of time and turns a Bullish Engulfing into
something that never occurs in real data, which would actively corrupt the labels. Only
augmentations that preserve the meaning of the chart are used — small translate/scale jitter
and mosaic, plus mild HSV jitter for colour robustness.

After every epoch the checkpoint is uploaded to the Hub, so a dropped session costs at most
one epoch.

In [ ]:
from ultralytics import YOLO

def upload_checkpoint(trainer):
    """Push last.pt to the Hub after each epoch so a disconnect is survivable."""
    last = pathlib.Path(trainer.last)
    if not last.exists():
        return
    try:
        api.upload_file(path_or_fileobj=str(last), path_in_repo='last.pt',
                        repo_id=MODEL_REPO, repo_type='model',
                        commit_message=f'checkpoint epoch {trainer.epoch + 1}')
    except Exception as exc:
        print('checkpoint upload failed (training continues):', exc)

model = YOLO(resume_from) if resume_from else YOLO('yolo11n.pt')
model.add_callback('on_fit_epoch_end', upload_checkpoint)

results = model.train(
    data=str(yaml_path),
    epochs=60,
    imgsz=640,
    batch=32,
    project='/content/runs/detect',
    name='train',
    exist_ok=True,
    resume=bool(resume_from),
    patience=15,
    seed=0,
    # --- augmentation: meaning-preserving only ---
    fliplr=0.0,        # a mirrored chart runs time backwards
    flipud=0.0,        # a flipped chart inverts price direction
    degrees=0.0,       # charts are never rotated in the wild
    perspective=0.0,
    shear=0.0,
    translate=0.05,
    scale=0.15,
    mosaic=0.5,
    close_mosaic=10,   # disable mosaic for the last 10 epochs
    hsv_h=0.010, hsv_s=0.3, hsv_v=0.2,
    plots=True,
)
print('training finished')

## 7. Evaluate on the held-out test split

Per-class numbers, not just one aggregate mAP. Class imbalance is severe — Doji outnumbers
Evening Star by more than an order of magnitude — so a single averaged figure would hide
both the classes that work and the ones that do not.

In [ ]:
import json

best = pathlib.Path('/content/runs/detect/train/weights/best.pt')
model = YOLO(str(best))
metrics = model.val(data=str(yaml_path), split='test', imgsz=640, verbose=True)

per_class = {}
for i, c in enumerate(CLASSES):
    try:
        p, r, ap50, ap = metrics.class_result(i)
        per_class[c] = {'precision': float(p), 'recall': float(r),
                        'mAP50': float(ap50), 'mAP50_95': float(ap)}
    except Exception:
        per_class[c] = None  # class absent from the test split

summary = {
    'aggregate': {'mAP50': float(metrics.box.map50), 'mAP50_95': float(metrics.box.map),
                  'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr)},
    'per_class': per_class,
    'split': 'test',
    'note': 'labels are TA-Lib rule-based, not human-verified',
}
print(json.dumps(summary, indent=2))
pathlib.Path('/content/detection_metrics.json').write_text(json.dumps(summary, indent=2))

## 8. Publish weights and metrics to the Hub

`best.pt` is what every later stage loads — the evaluation scripts, the downstream signal
study and the Gradio Space all pull this exact file, so they can never score a different
checkpoint than the one reported here.

In [ ]:
model_card = f'''---
license: mit
tags: [object-detection, yolo, ultralytics, candlestick, finance]
library_name: ultralytics
---

# Candlestick pattern detector (YOLO11n)

Detects {len(CLASSES)} candlestick patterns in 640x640 rendered daily charts of 20 candles.

## Labels are rule-based, not human-verified

Training boxes were generated by TA-Lib's `CDLxxx` functions. This model imitates that
heuristic; it does not reproduce a human analyst's judgement, and where TA-Lib and a trader
would disagree, this model follows TA-Lib.

## Classes

{chr(10).join(f'- `{i}` {c}' for i, c in enumerate(CLASSES))}

## Test-split results

| metric | value |
|---|---|
| mAP@50 | {summary['aggregate']['mAP50']:.4f} |
| mAP@50-95 | {summary['aggregate']['mAP50_95']:.4f} |
| precision | {summary['aggregate']['precision']:.4f} |
| recall | {summary['aggregate']['recall']:.4f} |

Splits are chronological with an embargo gap, never random.

Full code, per-class metrics and the downstream predictive-signal study:
https://github.com/{api.whoami()['name']}/candlestick-pattern-recognition-system
'''

api.upload_file(path_or_fileobj=str(best), path_in_repo='best.pt',
                repo_id=MODEL_REPO, repo_type='model', commit_message='trained weights')
api.upload_file(path_or_fileobj='/content/detection_metrics.json',
                path_in_repo='detection_metrics.json', repo_id=MODEL_REPO,
                repo_type='model', commit_message='test-split metrics')
api.upload_file(path_or_fileobj=model_card.encode(), path_in_repo='README.md',
                repo_id=MODEL_REPO, repo_type='model', commit_message='model card')

for extra in ('results.png', 'confusion_matrix_normalized.png', 'results.csv'):
    p = pathlib.Path('/content/runs/detect/train') / extra
    if p.exists():
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f'training/{extra}',
                        repo_id=MODEL_REPO, repo_type='model', commit_message=f'add {extra}')

print('\n=== DONE ===')
print(f'weights + metrics pushed to https://huggingface.co/{MODEL_REPO}')
print('You can close Colab and tell Claude Code to continue.')